In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import current_timestamp

In [0]:
df_paymentdetail = spark.read.format("delta").table("dev.tracking_br.paymentdetail")

In [0]:
from pyspark.sql.functions import col, to_date
df_paymentdetail = df_paymentdetail.select(col("id"),
                               col("srno").alias("serialno"),
                               col("paymentdate"),
                               col("amount").alias("paidamount"),
                               col("cheque").alias("chequeno"),
                               col("refno"),
                               col("payingparty"),
                               col("receivingparty"),
                               col("bankname"),
                               col("details"))

                               

In [0]:
from pyspark.sql.functions import current_timestamp
df_paymentdetail = df_paymentdetail.withColumn("ingestion_time",current_timestamp())
df_paymentdetail = convert_to_float(df_paymentdetail, "paidamount")
df_paymentdetail = convert_to_int(df_paymentdetail, "id")
df_paymentdetail = convert_to_timestamp(df_paymentdetail, "paymentdate")

In [0]:
df_paymentdetail.createOrReplaceTempView("temp_paymentdetail")

In [0]:
%sql
MERGE INTO dev.tracking_sl.paymentdetail tgt 
USING ( 
  SELECT DISTINCT * 
  FROM temp_paymentdetail 
) src 
ON tgt.id = src.id 
WHEN NOT MATCHED THEN 
  INSERT *